In [0]:
-- ============================================
-- QC — Page Views (Bronze)
-- ============================================

USE CATALOG shoply_analytics;
USE SCHEMA bronze;

-- 1. Preview
SELECT *
FROM page_views
LIMIT 20;

-- 2. Row count
-- 10,000
SELECT COUNT(*) AS row_count
FROM page_views;

-- 3. Duplicate session_id + timestamp + page_url
-- (page views rarely have a natural primary key)
SELECT session_id, timestamp, page_url, COUNT(*) AS cnt
-- 277 rows
FROM page_views
GROUP BY session_id, timestamp, page_url
HAVING COUNT(*) > 1;

-- 4. Null checks
-- 2183 user_id
-- 4819 campaign_id
-- 4005 referrer_url
-- 1000 timestamp
-- 237 timestamp
  SUM(CASE WHEN session_id IS NULL THEN 1 END) AS null_session_id,
  SUM(CASE WHEN user_id IS NULL THEN 1 END) AS null_user_id,
  SUM(CASE WHEN campaign_id IS NULL THEN 1 END) AS null_campaign_id,
  SUM(CASE WHEN page_url IS NULL THEN 1 END) AS null_page_url,
  SUM(CASE WHEN referrer_url IS NULL THEN 1 END) AS null_referrer_url,
  SUM(CASE WHEN timestamp IS NULL THEN 1 END) AS null_timestamp,
  SUM(CASE WHEN device_type IS NULL THEN 1 END) AS null_device_type,
  SUM(CASE WHEN country IS NULL THEN 1 END) AS null_country
FROM page_views;

-- 5. Malformed timestamps
-- ~1000+
SELECT *
FROM page_views
WHERE TRY_TO_TIMESTAMP(timestamp) IS NULL
  AND timestamp IS NOT NULL
  AND TRIM(timestamp) != '';

-- 6. Future timestamps
SELECT *
FROM page_views
WHERE TRY_TO_TIMESTAMP(timestamp) > current_timestamp();

-- 7. Unexpected device types
SELECT DISTINCT device_type
FROM page_views;

-- 8. Unwanted spaces in string columns
SELECT *
FROM page_views
WHERE session_id != TRIM(session_id)
   OR user_id != TRIM(user_id)
   OR campaign_id != TRIM(campaign_id)
   OR page_url != TRIM(page_url)
   OR referrer_url != TRIM(referrer_url)
   OR device_type != TRIM(device_type)
   OR country != TRIM(country);

-- 9. (not found in sessions table)
SELECT p.session_id
FROM page_views p
LEFT JOIN sessions s ON p.session_id = s.session_id
WHERE s.session_id IS NULL
  AND p.session_id IS NOT NULL;

-- 10. Column structure
DESCRIBE TABLE page_views;

